In [ ]:
# 14_embed_profiles.ipynb
#
# Embeds every natural-language profile from step 13 using the OpenAI
# text-embedding-3-small model (1 536 dimensions).
#
# Input:  data/13_nl_profiles/nl_profiles.csv   (pidp, ladcd, nl_profile)
# Output: data/14_embeddings/
#           embeddings.npy     — float32 array, shape (N, 1536)
#           embeddings_index.csv — pidp, ladcd, row index into the array
#           embeddings.parquet   — pidp, ladcd, embedding (list column)
#
# The .npy + index CSV is the most efficient format for downstream
# similarity search (FAISS, sklearn, etc.).
# The parquet is a convenience copy that keeps everything in one file.
#
# Requires OPENAI_API_KEY in environment or .env file.
# Embeddings are batched (512 texts per call) and checkpointed every
# CHECKPOINT_EVERY batches so long runs can be resumed after interruption.

import sys, os, time
sys.path.insert(0, os.path.abspath('..'))

import importlib
import data_pipeline.config_paths as _cp
importlib.reload(_cp)
from data_pipeline.config_paths import DATA_FOLDER, USE_FOUR_LA_SUBSET, FOUR_LA_CODES

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from openai import OpenAI

# ── Load .env if present ──────────────────────────────────────────────────────
try:
    from dotenv import load_dotenv
    load_dotenv(Path('..') / '.env', override=False)
    print("Loaded .env")
except ImportError:
    pass

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    raise EnvironmentError(
        "OPENAI_API_KEY not set. Export it in your shell or add it to .env:\n"
        "  export OPENAI_API_KEY=sk-..."
    )

# ── Config ────────────────────────────────────────────────────────────────────
EMBED_MODEL       = "text-embedding-3-small"   # 1 536 dims, $0.02 / 1M tokens
EMBED_DIMS        = 1_536
BATCH_SIZE        = 512    # texts per API call (max 2 048)
CHECKPOINT_EVERY  = 10     # save partial results every N batches
RETRY_LIMIT       = 3
RETRY_DELAY       = 5      # seconds between retries

INPUT_CSV  = Path(f"../{DATA_FOLDER}/13_nl_profiles/nl_profiles.csv")
OUTPUT_DIR = Path(f"../{DATA_FOLDER}/14_embeddings")
OUTPUT_NPY     = OUTPUT_DIR / "embeddings.npy"
OUTPUT_IDX_CSV = OUTPUT_DIR / "embeddings_index.csv"
OUTPUT_PARQUET = OUTPUT_DIR / "embeddings.parquet"
CHECKPOINT_NPY = OUTPUT_DIR / "_checkpoint.npy"
CHECKPOINT_IDX = OUTPUT_DIR / "_checkpoint_idx.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = OpenAI(api_key=OPENAI_API_KEY)

# ── Load profiles ─────────────────────────────────────────────────────────────
print(f"Reading: {INPUT_CSV}")
df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df):,} profiles")

# Apply LA filter consistent with earlier steps
if USE_FOUR_LA_SUBSET:
    df = df[df["ladcd"].isin(set(FOUR_LA_CODES))].reset_index(drop=True)
    print(f"Filtered to {len(df):,} profiles (4-LA subset)")

# Drop rows with empty profiles
n_before = len(df)
df = df[df["nl_profile"].notna() & (df["nl_profile"].str.strip() != "")].reset_index(drop=True)
if len(df) < n_before:
    print(f"Dropped {n_before - len(df):,} rows with empty profiles")

texts   = df["nl_profile"].tolist()
n_total = len(texts)
n_batches = (n_total + BATCH_SIZE - 1) // BATCH_SIZE

# Rough cost estimate
avg_tokens = 80
est_tokens = n_total * avg_tokens
est_cost   = est_tokens / 1_000_000 * 0.02
print(f"\nModel:          {EMBED_MODEL}")
print(f"Profiles:       {n_total:,}")
print(f"Batches:        {n_batches:,}  (batch size {BATCH_SIZE})")
print(f"Est. tokens:    ~{est_tokens:,.0f}")
print(f"Est. cost:      ~${est_cost:.3f} USD")

In [ ]:
# ── Resume from checkpoint if one exists ─────────────────────────────────────
if CHECKPOINT_NPY.exists() and CHECKPOINT_IDX.exists():
    chk_emb = np.load(CHECKPOINT_NPY)
    chk_idx = pd.read_csv(CHECKPOINT_IDX)
    done_count = len(chk_idx)
    print(f"Resuming from checkpoint: {done_count:,} profiles already embedded")
    embeddings_list = list(chk_emb)
    start_idx = done_count
else:
    embeddings_list = []
    start_idx = 0
    print("Starting fresh")


def embed_batch(batch_texts: list[str], attempt: int = 0) -> list[list[float]]:
    """Call the embeddings API for a batch of texts; retries on transient errors."""
    try:
        response = client.embeddings.create(
            input=batch_texts,
            model=EMBED_MODEL,
        )
        return [item.embedding for item in response.data]
    except Exception as exc:
        if attempt < RETRY_LIMIT:
            print(f"  API error ({exc}); retrying in {RETRY_DELAY}s ...")
            time.sleep(RETRY_DELAY)
            return embed_batch(batch_texts, attempt + 1)
        raise


# ── Embed ─────────────────────────────────────────────────────────────────────
remaining_texts = texts[start_idx:]
remaining_batches = (len(remaining_texts) + BATCH_SIZE - 1) // BATCH_SIZE

with tqdm(total=len(remaining_texts), desc="Embedding", unit="profile") as pbar:
    for batch_num, batch_start in enumerate(range(0, len(remaining_texts), BATCH_SIZE)):
        batch = remaining_texts[batch_start : batch_start + BATCH_SIZE]
        vecs  = embed_batch(batch)
        embeddings_list.extend(vecs)
        pbar.update(len(batch))

        # Checkpoint periodically
        if (batch_num + 1) % CHECKPOINT_EVERY == 0:
            chk_arr = np.array(embeddings_list, dtype=np.float32)
            np.save(CHECKPOINT_NPY, chk_arr)
            df.iloc[: len(embeddings_list)][["pidp", "ladcd"]].to_csv(CHECKPOINT_IDX, index=False)
            pbar.set_postfix({"checkpoint": len(embeddings_list)})

print(f"\nDone — {len(embeddings_list):,} embeddings collected")

In [ ]:
# ── Save outputs ──────────────────────────────────────────────────────────────
embeddings = np.array(embeddings_list, dtype=np.float32)
assert embeddings.shape == (len(df), EMBED_DIMS), (
    f"Shape mismatch: {embeddings.shape} vs expected ({len(df)}, {EMBED_DIMS})"
)

# 1. Numpy array
np.save(OUTPUT_NPY, embeddings)
print(f"Saved numpy array: {OUTPUT_NPY}  shape={embeddings.shape}")

# 2. Index CSV  (row i of embeddings.npy corresponds to row i of this CSV)
index_df = df[["pidp", "ladcd"]].copy()
index_df["embedding_row"] = index_df.index
index_df.to_csv(OUTPUT_IDX_CSV, index=False)
print(f"Saved index CSV:   {OUTPUT_IDX_CSV}")

# 3. Parquet with embedding as a list column (convenient for pandas workflows)
out_df = df[["pidp", "ladcd", "nl_profile"]].copy()
out_df["embedding"] = [row.tolist() for row in embeddings]
out_df.to_parquet(OUTPUT_PARQUET, index=False)
print(f"Saved parquet:     {OUTPUT_PARQUET}")

# Clean up checkpoint files
for f in [CHECKPOINT_NPY, CHECKPOINT_IDX]:
    if f.exists():
        f.unlink()

print(f"\nAll done. Embedding matrix: {embeddings.shape}")
print(f"  dtype={embeddings.dtype}  size on disk ≈ {embeddings.nbytes / 1e6:.1f} MB (uncompressed)")